# Headless Data Visualization (HPC / no-display)

This notebook is the headless-safe counterpart to `00_data_visualization.ipynb`. That
notebook uses napari, which requires a Qt display and cannot run on headless SLURM
compute nodes; it also only loads the raw brightfield (BF) channel.

This notebook instead:
- loads BF together with its corresponding inference segmentation mask and its
  corresponding mCherry channel for the same well/timepoint/z-slice,
- renders them as independent, toggleable, opacity-adjustable layers (like napari's
  layer panel) using matplotlib (`Agg` backend) + `ipywidgets`, and
- lets you browse across wells and timepoints.

Use `00_data_visualization.ipynb` instead if you're on a local machine with a display
and want full napari interactivity (3D rendering, arbitrary colormaps, etc.).

In [ ]:
from pathlib import Path

import tifffile
from IPython.display import display

from src.inference.output_manager import OutputManager
from src.utils.file_utils import ConfigurableFileHandler
from src.visualize import (
    build_browser_widget,
    build_index,
    build_layer_controls,
    render_layers,
    resolve_related_paths,
)

# --- Notebook-level constants (edit these for your dataset) ---
# Mirrors config.yaml's paths.input_dir
RAW_DATA_DIR = Path("data/sample_data")
# Mirrors config.yaml's paths.output_dir / segmentation.inference.results_folder
INFERENCE_OUTPUT_DIR = Path("data/sample_data_output") / "inference"
# Mirrors config.yaml's segmentation.cellpose.model_type
MODEL_NAME = "cellpose_sam"
# Mirrors config.yaml's segmentation.inference.dataset_name
DATASET_NAME = "test"
# Mirrors config.yaml's segmentation.inference.label_format
LABEL_FORMAT = "zarr"

file_handler = ConfigurableFileHandler()
output_manager = OutputManager(
    base_output_dir=INFERENCE_OUTPUT_DIR,
    model_name=MODEL_NAME,
    dataset_name=DATASET_NAME,
    label_format=LABEL_FORMAT,
)

## Single well/timepoint QC view

In [ ]:
# Replace with a specific file path if you don't want the first match, e.g.:
# bf_path = RAW_DATA_DIR / "p2126_A01_t1_z1_BF.tif"
bf_path = next(RAW_DATA_DIR.glob("*_BF.tif"))

paths = resolve_related_paths(bf_path, output_manager)

bf = tifffile.imread(paths["bf"])
mcherry = tifffile.imread(paths["mcherry"]) if paths["mcherry"] is not None else None
mask = OutputManager.load_masks(paths["mask"]) if paths["mask"] is not None else None

fig, layer_artists = render_layers(bf, mask=mask, mcherry=mcherry)
display(fig)
display(build_layer_controls(layer_artists, fig))

## Browse across wells/timepoints

In [ ]:
index = build_index(RAW_DATA_DIR, file_handler)
display(build_browser_widget(index, output_manager))

## Static / batch-safe example (no widgets)

This cell has no `ipywidgets` dependency — it's what you'd run under
`papermill`/`sbatch` where no one is watching interactively. It saves a PNG instead of
displaying inline.

In [ ]:
fig, _ = render_layers(bf, mask=mask, mcherry=mcherry)
fig.savefig("qc_view.png", dpi=150, bbox_inches="tight")